# ETH ICT-Macro / PA-Microstructure Portfolio Audit

This notebook reads the frozen research outputs. It does not tune model parameters. The live decision gate is **PAPER_TRADE_ONLY / NO_LIVE_EDGE_APPROVED** because CAGR is below maximum drawdown.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'ict_pa_v1' / 'results').exists():
    ROOT = ROOT / 'research' / 'eth_ict_price_action_portfolio'
RESULTS = ROOT / 'ict_pa_v1' / 'results'
summary = pd.read_csv(RESULTS / 'summary.csv')
priority = pd.read_csv(RESULTS / 'final_priority_metrics.csv')
daily = pd.read_csv(RESULTS / 'daily_equity.csv', parse_dates=['date'])
yearly = pd.read_csv(RESULTS / 'yearly.csv')
segments = pd.read_csv(RESULTS / 'validation_segments.csv')

## User-priority scorecard

All streaks are computed on natural-day account data. A flat day has zero gross exposure for every 15-minute interval in that day.

In [ ]:
priority

## Fixed cost and execution stresses

In [ ]:
summary[['scenario', 'total_return', 'cagr', 'max_drawdown', 'calmar', 'liquidation_events']]

## Daily equity and drawdown landmarks

The curve is positive overall but does not satisfy the requested stability ratio.

In [ ]:
landmarks = daily.set_index('date')[['equity', 'drawdown', 'max_gross_exposure']].resample('QS').last()
landmarks

## Reproducibility checks

In [ ]:
base = summary.loc[summary['scenario'].eq('base')].iloc[0]
assert priority.loc[0, 'max_consecutive_flat_days'] == 0
assert priority.loc[0, 'max_consecutive_losing_days'] == 9
assert base['liquidation_events'] == 0
assert base['max_gross_exposure'] <= 1.0
assert abs(base['total_return'] - (daily['equity'].iloc[-1] - 1.0)) < 1e-10
assert not bool(priority.loc[0, 'passes_cagr_ge_drawdown'])
'All frozen artifact checks passed.'